# Transformer (COMING SOON)

In this section, we will generate forecasts of EEG activity using a Transformer-based model. The VARIMA, DMD, and TCN models primarly focus on modeling short-term temporal correlations within a system and struggle to capture the global dynamics present within long-term time windows. Transformers, on the other hand, excel at modeling the long-range dependencies between samples of a sequence—making them a promising approach to characterize and forecast the multi-scale temporal structures present in neural activity.    

## Background

The Transformer architecture was introduced in the seminal paper by Vaswani et al. (2017), which demonstrated that attention mechanisms alone were sufficient for solving complex sequence modeling and transduction problems. This architecture represented a fundamental shift in deep learning by dispensing with the recursive constraints of traditional models (e.g., recurrent networks). By processing entire sequences simultaneously rather than step by step, the Transformer enables the modeling of lengthy sequences that were previously computationally prohibitive.

Transformers have exhibited unparalleled success within the field of Natural Language Processing (NLP). In tasks such as machine translation, this architecture serves as a core component of cutting-edge Large Language Models (LLMs) that effectively capturing the semantic relationships between words and concepts, irrespective of there distance within an excerpt.

### Queries, Keys, and Values

At the core of the Transformer is the **self-attention mechanism**, which allows the model to decide how much each position in a sequence should "look at" every other position when building its representation.

To build intuition, consider a library search: you type a **query** describing what you want, the library matches it against the **keys** (catalog index entries) of every book on the shelf, and the books with the closest-matching keys are retrieved—their **values** (the actual content) are then blended together in proportion to how well they matched your query. Self-attention works the same way, except every position in the sequence simultaneously acts as both a searcher (submitting a query) *and* a book (advertising a key and holding a value).

Concretely, given an input matrix $X \in \mathbb{R}^{T \times d_{\text{model}}}$, where $T$ is the sequence length and $d_{\text{model}}$ is the embedding dimension, three separate linear projections produce the query, key, and value matrices:

$$Q = XW^Q, \quad K = XW^K, \quad V = XW^V$$

where $W^Q, W^K, W^V \in \mathbb{R}^{d_{\text{model}} \times d_k}$ are learned weight matrices and $d_k$ is the projection dimension. Because all three matrices are derived from the same input $X$, the mechanism is called **self**-attention—each position queries and is queried by every other position in the same sequence.

### Scaled Dot-Product Attention

Attention scores are computed by measuring the similarity between each query and every key via a scaled dot product. The resulting scores are normalized with a softmax so they sum to one, then used to form a weighted sum of the values:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

The division by $\sqrt{d_k}$ is a stabilizing trick: when $d_k$ is large the dot products tend to grow large in magnitude, pushing the softmax into regions where its gradients nearly vanish. Scaling counters this. The softmax output is an attention weight matrix $A \in \mathbb{R}^{T \times T}$, where entry $A_{ij}$ encodes how strongly time step $i$ attends to time step $j$.

Rather than performing a single attention operation, the Transformer employs **multi-head attention**, running $h$ independent attention operations in parallel across different learned subspaces and concatenating their outputs:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\,W^O, \quad \text{head}_i = \text{Attention}(QW_i^Q,\; KW_i^K,\; VW_i^V)$$

where $W^O \in \mathbb{R}^{hd_k \times d_{\text{model}}}$ is a learned output projection. Each head is free to specialize in a different kind of relationship—one head might track short-range structure while another captures longer-range dependencies—giving the model far richer representational capacity than a single attention operation.

### Autoregressive Prediction

The original Transformer follows an **encoder–decoder** design. The encoder reads the full input sequence and produces a context representation via self-attention. The decoder then generates the output sequence one step at a time, using **cross-attention** to consult that context at each step.

Cross-attention operates the same way as self-attention, but the queries and keys/values come from *different* sequences. Specifically, the decoder's current output supplies the queries ($Q$), while the encoder's context supplies the keys and values ($K$, $V$):

$$\text{CrossAttention}(Q_\text{dec},\, K_\text{enc},\, V_\text{enc}) = \text{softmax}\!\left(\frac{Q_\text{dec}\,K_\text{enc}^\top}{\sqrt{d_k}}\right)V_\text{enc}$$

This lets each decoder position ask: *"which parts of the input sequence are most relevant to what I am about to predict?"* and retrieve the corresponding encoder context.

During training, the decoder also applies **masked self-attention** over its own outputs: future positions are hidden by setting their attention scores to $-\infty$ before the softmax, so the model can only attend to steps it has already generated. This ensures the model learns to predict each step from the past alone—without peeking at the future.

At inference, predictions are generated **autoregressively**: the model predicts step $t+1$, appends it to the decoder input, predicts step $t+2$, and so on until the target horizon $H$ is reached:

$$\hat{x}_{t+h} = f_\theta\!\left(x_1, \ldots, x_t,\; \hat{x}_{t+1}, \ldots, \hat{x}_{t+h-1}\right), \quad h = 1, \ldots, H$$

Because each generated step is fed back as input for the next, any prediction error accumulates over the horizon—a phenomenon known as **exposure bias**. This compounding makes long-horizon forecasting substantially harder than single-step prediction, and is one of the primary motivations for the time series adaptations discussed below.

### From NLP to Time Series Forecasting

Adapting the Transformer from discrete language tokens to continuous-valued time series requires several modifications.

**Input representation.** In NLP, tokens are drawn from a finite vocabulary and embedded via a lookup table. Time series observations are continuous, so a learned linear projection is used instead to map each time step's values into the model dimension. For multivariate recordings like EEG, the input at each step is a vector $\mathbf{x}_t \in \mathbb{R}^C$ across $C$ channels.

**Positional encoding.** Because self-attention is permutation-invariant—it has no built-in notion of order—a positional encoding is added to each embedding to inject information about where in the sequence a sample falls. NLP models use sinusoidal encodings or learned absolute positions. Time series models often enrich this with domain-specific temporal features such as time-of-day, frequency band, or trial index.

**Efficient attention for long sequences.** Computing $QK^\top$ requires a dot product between every pair of time steps, producing an attention matrix of size $T \times T$. Both the computation and memory therefore scale as $\mathcal{O}(T^2)$—doubling the sequence length quadruples the cost. For short sentences this is acceptable, but for long high-frequency signals like EEG (e.g., $T = 1000$ steps at 256 Hz), the attention matrix alone contains $10^6$ entries, making full attention computationally prohibitive.

Several architectures address this:

- **FEDformer** (Zhou et al., 2022) performs attention in the frequency domain using a sparse Fourier or wavelet decomposition. By attending over spectral components rather than individual time steps, it efficiently captures the global periodicities and trends that are particularly prominent in neural oscillations.

- **PatchTST** (Nie et al., 2023) divides the input into non-overlapping patches and treats each patch as a token, dramatically shortening the effective sequence length and improving long-horizon accuracy.

- **Direct multi-step prediction head.** Rather than decoding one step at a time as in NLP, many time series Transformers attach a linear projection head to the encoder output that maps the full context representation to all $H$ future steps simultaneously, sidestepping the exposure bias inherent in autoregressive decoding.

- **Informer** (Zhou et al., 2021) introduces *ProbSparse* attention. The key observation is that in practice, each query's attention distribution is highly uneven: a small number of keys dominate and the rest receive near-zero weight. Informer identifies the most *informative* queries—those with the most peaked distributions—by scoring each query $q_i$ against a random sample of $\ln T$ keys:

$$\bar{M}(q_i, K) = \max_{j} \left\{\frac{q_i k_j^\top}{\sqrt{d_k}}\right\} - \frac{1}{T}\sum_{j=1}^{T} \frac{q_i k_j^\top}{\sqrt{d_k}}$$

  A large $\bar{M}$ means the query's attention is concentrated (informative); a small value means it is nearly uniform (uninformative). Only the top-$u = c \ln T$ queries by this score are selected to participate in full attention, while all remaining queries default to the mean of $V$:

$$\text{ProbSparseAttn}(\bar{Q}, K, V) = \text{softmax}\!\left(\frac{\bar{Q}K^\top}{\sqrt{d_k}}\right)V$$

  where $\bar{Q} \in \mathbb{R}^{u \times d_k}$ is the selected query subset. This reduces complexity to $\mathcal{O}(T \log T)$ in both time and memory.

In this tutorial, we implement the **Informer**, whose ProbSparse attention mechanism makes it well-suited to the long, high-frequency nature of EEG recordings.